# Compare MVP1 and MVP2 portrait framing

This notebook runs the **actual command-line apply scripts** for every supported image directly inside top-level `Test/`; it does not reproduce either framing algorithm.

- **MVP1** applies its learned canonical alpha template and black exterior. In its default mode, a non-260×310 input is resized directly to 260×310.
- **MVP2** renders a scalar procedural rounded frame and halo. It normally rejects other dimensions, so this batch explicitly supplies `--resize`; the current 253×336 input is resized directly to 260×310.

Outputs are overwritten deterministically on rerun. Generated folders are outside `Test/`, so generated images never become inputs.


## Configuration and project discovery


In [1]:
from pathlib import Path
import subprocess
import sys
from PIL import Image, ImageDraw

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
TARGET_SIZE = (260, 310)

def find_project_root(start: Path) -> Path:
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "MVP1" / "apply_border.py").is_file() and (candidate / "MVP2" / "apply_frame.py").is_file():
            return candidate
    raise FileNotFoundError(f"Could not find a project root above {resolved} containing both apply scripts")

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / "Test"
MVP1_DIR = PROJECT_ROOT / "TestOutputs" / "MVP1"
MVP2_DIR = PROJECT_ROOT / "TestOutputs" / "MVP2"
COMPARISON_DIR = PROJECT_ROOT / "TestOutputs" / "Comparisons"
for directory in (MVP1_DIR, MVP2_DIR, COMPARISON_DIR):
    directory.mkdir(parents=True, exist_ok=True)

inputs = sorted(
    (path for path in INPUT_DIR.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS),
    key=lambda path: path.name.lower(),
)
if not inputs:
    raise RuntimeError(f"No supported top-level images found in {INPUT_DIR}")

def output_name(path: Path) -> str:
    return path.name if path.suffix.lower() == ".png" else f"{path.name}.png"

print(f"Project root: {PROJECT_ROOT}")
print(f"Interpreter: {sys.executable}")
print(f"Inputs ({len(inputs)}): {', '.join(path.name for path in inputs)}")


Project root: D:\Random\Pic
Interpreter: C:\Python314\python.exe
Inputs (1): alavez.png


## Batch execution


In [2]:
records = []
for input_path in inputs:
    mvp1_path = MVP1_DIR / output_name(input_path)
    mvp2_path = MVP2_DIR / output_name(input_path)
    commands = {
        "MVP1": [sys.executable, str(PROJECT_ROOT / "MVP1" / "apply_border.py"), str(input_path), str(mvp1_path)],
        "MVP2": [sys.executable, str(PROJECT_ROOT / "MVP2" / "apply_frame.py"), str(input_path), str(mvp2_path), "--resize"],
    }
    results = {}
    for method, command in commands.items():
        completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
        results[method] = completed
        if completed.returncode != 0:
            raise RuntimeError(
                f"{method} failed for {input_path.name} with exit code {completed.returncode}\n"
                f"Command: {command!r}\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}"
            )
        print(f"{method} / {input_path.name}: {completed.stdout.strip()}")
    records.append({"input": input_path, "mvp1": mvp1_path, "mvp2": mvp2_path, "results": results})


MVP1 / alavez.png: Saved D:\Random\Pic\TestOutputs\MVP1\alavez.png  (260x310, mode=RGBA, fit=False)
MVP2 / alavez.png: Saved D:\Random\Pic\TestOutputs\MVP2\alavez.png (260x310, RGBA)


## Validation summary


In [3]:
validation_rows = []
for record in records:
    for method, path in (("MVP1", record["mvp1"]), ("MVP2", record["mvp2"])):
        if not path.is_file():
            raise AssertionError(f"{method} output does not exist: {path}")
        with Image.open(path) as image:
            actual_size, actual_mode = image.size, image.mode
            image.verify()
        if actual_size != TARGET_SIZE:
            raise AssertionError(f"{method} output has size {actual_size}, expected {TARGET_SIZE}: {path}")
        if actual_mode != "RGBA":
            raise AssertionError(f"{method} output has mode {actual_mode}, expected RGBA: {path}")
        validation_rows.append((record["input"].name, method, record["results"][method].returncode, path.exists(), actual_size, actual_mode))

print("input | method | return | exists | size | mode")
print("-" * 62)
for name, method, returncode, exists, size, mode in validation_rows:
    print(f"{name} | {method} | {returncode} | {exists} | {size[0]}x{size[1]} | {mode}")
print(f"PASS: validated {len(validation_rows)} transformed images from {len(records)} input(s).")


input | method | return | exists | size | mode
--------------------------------------------------------------
alavez.png | MVP1 | 0 | True | 260x310 | RGBA
alavez.png | MVP2 | 0 | True | 260x310 | RGBA
PASS: validated 2 transformed images from 1 input(s).


## Visual side-by-side comparisons

Transparency is composited over a checkerboard. Each contact sheet is saved as a PNG and displayed inline when IPython is available.


In [4]:
def checkerboard(size, tile=12, colors=((224, 224, 224), (176, 176, 176))):
    board = Image.new("RGB", size, colors[0])
    draw = ImageDraw.Draw(board)
    for y in range(0, size[1], tile):
        for x in range(0, size[0], tile):
            if (x // tile + y // tile) % 2:
                draw.rectangle((x, y, min(x + tile - 1, size[0] - 1), min(y + tile - 1, size[1] - 1)), fill=colors[1])
    return board

def on_checkerboard(path, size=TARGET_SIZE):
    with Image.open(path) as source:
        rgba = source.convert("RGBA")
        if rgba.size != size:
            rgba.thumbnail(size, Image.Resampling.LANCZOS)
            layer = Image.new("RGBA", size, (0, 0, 0, 0))
            layer.alpha_composite(rgba, ((size[0] - rgba.width) // 2, (size[1] - rgba.height) // 2))
            rgba = layer
    board = checkerboard(size).convert("RGBA")
    board.alpha_composite(rgba)
    return board.convert("RGB")

def make_contact_sheet(record):
    labels = ("Input", "MVP1", "MVP2")
    paths = (record["input"], record["mvp1"], record["mvp2"])
    panel_width, panel_height, label_height, gap = 260, 310, 30, 12
    sheet = Image.new("RGB", (3 * panel_width + 4 * gap, panel_height + label_height + 2 * gap), "white")
    draw = ImageDraw.Draw(sheet)
    for index, (label, path) in enumerate(zip(labels, paths)):
        x = gap + index * (panel_width + gap)
        draw.text((x + 4, 7), label, fill="black")
        sheet.paste(on_checkerboard(path), (x, gap + label_height))
    return sheet

contact_sheets = []
for record in records:
    comparison_path = COMPARISON_DIR / f"{record['input'].name}_comparison.png"
    sheet = make_contact_sheet(record)
    sheet.save(comparison_path, format="PNG")
    contact_sheets.append(comparison_path)
    print(f"Saved {comparison_path}")
    try:
        from IPython.display import display
        display(sheet)
    except ImportError:
        print("Inline display unavailable (IPython not installed); contact sheet saved above.")


Saved D:\Random\Pic\TestOutputs\Comparisons\alavez.png_comparison.png
Inline display unavailable (IPython not installed); contact sheet saved above.
